#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
audit_api_data.py — 상병 데이터(data_*.pq)의 완전성·정합성 점검과 분류별 사용 등급 판정

보유 분류 데이터(meft_*.csv)와 API 수집분을 (진료년월 × 약효분류) 단위로 대조한다.
  1) 커버리지 = 상병 합계 ÷ (보유 값 × 단위 환산계수). 환산계수는 비율 중앙값에 가장 가까운 10의 거듭제곱(--scale-amt/--scale-qty로 지정 가능)
  2) 분류별 커버리지의 수준·안정성(월별 변동계수)·월 누락·잘림 의심으로 사용 등급을 매긴다
       A    : 커버리지 ≥ 98%, 변동계수 ≤ 2%, 월 누락 없음 → 상병 귀속·상병 시장 총량에 사용
       B    : 커버리지 ≥ 90%, 변동계수 ≤ 5%            → 상병 구성(비중·추세)에 사용, 미귀속 잔여 항을 함께 표기
       C    : 그 외(저조·불안정·월 누락·잘림·시기별 변화) → 상병 귀속 보류, 원인 규명 후 재수집 대상
            (시기별 변화: 초기 25% 구간 대비 최근 25% 구간 커버리지 차이가 3%p 초과)
       미수집: API 행이 전혀 없음                       → 로그로 원인 구분(실패/응답 없음/미시도)
  3) 귀속 신뢰비: 커버리지가 기간 중 바뀌면 상병 합계의 성장률이 총계 성장률에서 벗어난다.
       오차 ≈ (최근 연도 커버리지 ÷ 첫 연도 커버리지 − 1), 귀속 신뢰비 = |오차| ÷ max(|분류 성장률|, 5%).
       0.2 미만이면 커버리지 변화가 분류 성장의 20% 미만이므로 상병 귀속을 신뢰한다 (초기 기준).
  4) 상병 귀속 사용 범위(운영 라벨):
       UNAVAILABLE : 해당 분류의 상병 데이터가 전혀 없음 → 상병 귀속 제외 (분류 총량 분석에는 계속 사용), 제외 분류의 시장 비중을 표시
       USABLE      : 미귀속률 ≤ --max-unattributed(기본 4%, 운영 임계값) 이고 시기별로 안정(귀속 신뢰비 < 0.2, 월 누락·잘림 없음)
       LIMITED     : 수집은 되지만 위 조건 미달 → 귀속 분석은 주의 표기 또는 제외
       상병 금액을 100%에 맞추려고 배율을 곱하는 보정은 하지 않는다. 관측된 상병 금액 + 미귀속 잔여항으로 그대로 둔다.
  5) 연도별 가중 커버리지(잔여 비중의 시기별 변화), 보험구분코드 분포, 상병 코드 형식 확인
  4) 수량은 같은 방식으로 요약만 한다 (수량은 분석 판정에 사용하지 않음)

입력  : data_*.pq (컬럼: 진료년월, 약효분류번호, 상병코드, 의약품사용금액, 총사용량, [보험구분코드]), meft_*.csv, [log.csv]
출력  : <out>/audit_report.md, <out>/audit_class_tiers.csv, <out>/audit_by_class_month.csv

실행  : python audit_api_data.py --api-dir api_data --csv meft_2014-2025.csv --out audit [--log api_data/log.csv]
"""


In [ ]:
# -*- coding: utf-8 -*-
"""2015~2016 meft vs sick(상병) 합산 커버리지 비교 (노트북용)"""
from pathlib import Path
import numpy as np
import pandas as pd

YEARS = ("2015", "2016","2020","2021","2022","2023","2024","2025")   # ★ 여기만 2015, 2016


# ---------------- 로딩 ----------------
def load_api(api_dir: Path) -> pd.DataFrame:
    api_dir = Path(api_dir)
    files = sorted((api_dir / "sick").glob("data_*.pq"))
    if not files:
        files = sorted((api_dir / "sick").glob("*.pq"))
    if not files:
        files = sorted(api_dir.rglob("*.pq"))
    if not files:
        print("[진단] 실제 파일:")
        for p in sorted(api_dir.rglob("*")):
            print("   ", p)
        raise SystemExit(f"{api_dir} 에서 pq 못 찾음")
    print(f"[탐색] pq {len(files)}개:")
    # for f in files:
    #     print("   ", f)

    parts = []
    for f in files:
        d = pd.read_parquet(f)
        d["진료년월"] = d["진료년월"].astype(str).str.replace(r"\D", "", regex=True).str[:6]
        d = d[d["진료년월"].str[:4].isin(YEARS)]
        if d.empty:
            continue
        d["code"] = pd.to_numeric(d["약효분류번호"], errors="coerce")
        g = (
            d.groupby(["진료년월", "code"])
            .agg(
                api_amt=("의약품사용금액", "sum"),
                api_qty=("총사용량", "sum"),
                rows=("상병코드", "size"),
                n_dx=("상병코드", "nunique"),
            )
            .reset_index()
        )
        parts.append(g)
    if not parts:
        raise SystemExit("pq 찾았지만 데이터 없음")
    out = pd.concat(parts, ignore_index=True)
    out["code"] = out["code"].astype("Int64")
    return out


def load_base(csv: Path) -> pd.DataFrame:
    b = pd.read_csv(csv, encoding="utf-8-sig")
    b = b.rename(columns={
        "약효분류코드": "code",
        "진료년월": "ym",
        "수량_천개": "base_qty",
        "사용금액_백만원": "base_amt",
    })
    b["진료년월"] = b["ym"].astype(str).str.replace(r"\D", "", regex=True).str[:6]
    b = b[b["진료년월"].str[:4].isin(YEARS)]
    b["code"] = pd.to_numeric(b["code"], errors="coerce").astype("Int64")
    return b[["진료년월", "code", "base_amt", "base_qty"]].dropna(subset=["code"])


# ---------------- 유틸 ----------------
def pick_scale(ratio: pd.Series) -> float:
    r = ratio.replace([np.inf, -np.inf], np.nan).dropna()
    r = r[r > 0]
    return float(10 ** round(np.log10(r.median()))) if len(r) else 1.0


def fmt_pct(x):
    return "-" if pd.isna(x) else f"{x:.2%}"


def fmt_num(x):
    return "-" if pd.isna(x) else f"{x:,.0f}"


# ---------------- 본체 ----------------
def run(api_dir=".", csv="meft_2014-2025.csv", out="audit_1516",
        min_base_amt=100.0, scale_amt=None, scale_qty=None, top=30):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)

    api = load_api(Path(api_dir))
    base = load_base(Path(csv))
    print(f"[로드] API {len(api):,}행 / meft {len(base):,}행")
    print(f"[월] API : {sorted(api['진료년월'].unique())}")
    print(f"[월] meft: {sorted(base['진료년월'].unique())}")

    m = base.merge(api, on=["진료년월", "code"], how="outer", indicator=True)

    both = m[m["_merge"] == "both"].copy()
    ref = both[(both["base_amt"] >= min_base_amt) & (both["api_amt"] > 0)]
    scale_amt = scale_amt or pick_scale(ref["api_amt"] / ref["base_amt"])
    refq = both[(both["base_qty"] >= 100) & (both["api_qty"] > 0)]
    scale_qty = scale_qty or pick_scale(refq["api_qty"] / refq["base_qty"])
    print(f"[환산계수] 금액 {scale_amt:g}, 수량 {scale_qty:g}")

    m["base_amt_won"] = m["base_amt"] * scale_amt
    m["base_qty_std"] = m["base_qty"] * scale_qty
    m["cov_amt"] = m["api_amt"] / m["base_amt_won"].where(m["base_amt_won"] > 0)
    m["cov_qty"] = m["api_qty"] / m["base_qty_std"].where(m["base_qty_std"] > 0)

    base_amt_sum = m["base_amt_won"].fillna(0).sum()
    api_amt_sum = m["api_amt"].fillna(0).sum()
    base_qty_sum = m["base_qty_std"].fillna(0).sum()
    api_qty_sum = m["api_qty"].fillna(0).sum()
    cov_all_amt = api_amt_sum / base_amt_sum if base_amt_sum else np.nan
    cov_all_qty = api_qty_sum / base_qty_sum if base_qty_sum else np.nan

    by_code = (
        m.groupby("code", dropna=False)
        .agg(
            base_amt_won=("base_amt_won", "sum"),
            api_amt=("api_amt", "sum"),
            base_qty_std=("base_qty_std", "sum"),
            api_qty=("api_qty", "sum"),
            months_base=("base_amt_won", lambda x: int((x > 0).sum())),
            months_api=("api_amt", lambda x: int(x.notna().sum())),
            api_rows=("rows", "sum"),
        )
        .reset_index()
    )
    by_code["cov_amt"] = by_code["api_amt"] / by_code["base_amt_won"].where(by_code["base_amt_won"] > 0)
    by_code["cov_qty"] = by_code["api_qty"] / by_code["base_qty_std"].where(by_code["base_qty_std"] > 0)
    by_code["unattributed_amt"] = 1 - by_code["cov_amt"]
    by_code["share_of_base"] = by_code["base_amt_won"] / by_code["base_amt_won"].sum()
    by_code["uncovered_share_of_base"] = by_code["share_of_base"] * by_code["unattributed_amt"].clip(lower=0)
    by_code = by_code.sort_values("base_amt_won", ascending=False)
    by_code.to_csv(out / "audit_by_class.csv", index=False, encoding="utf-8-sig")

    by_month = (
        m.groupby("진료년월")
        .agg(base_amt_won=("base_amt_won", "sum"),
             api_amt=("api_amt", "sum"),
             base_qty_std=("base_qty_std", "sum"),
             api_qty=("api_qty", "sum"))
        .reset_index()
        .sort_values("진료년월")
    )
    by_month["cov_amt"] = by_month["api_amt"] / by_month["base_amt_won"].where(by_month["base_amt_won"] > 0)
    by_month["cov_qty"] = by_month["api_qty"] / by_month["base_qty_std"].where(by_month["base_qty_std"] > 0)
    by_month.to_csv(out / "audit_by_month.csv", index=False, encoding="utf-8-sig")

    n_both = int((m["_merge"] == "both").sum())
    n_left = int((m["_merge"] == "left_only").sum())
    n_right = int((m["_merge"] == "right_only").sum())

    L = []
    L.append("# 2015~2016년 meft vs sick(상병) 데이터 비교\n")
    L.append("## 0. 데이터 개요\n")
    L.append(f"- API(sick) 행(월×분류): {len(api):,}")
    L.append(f"- meft(보유) 행(월×분류): {len(base):,}")
    L.append(f"- 병합: 양쪽 {n_both:,} / meft만 {n_left:,} / API만 {n_right:,}")
    L.append(f"- 대상 월: {len(by_month)}개 ({by_month['진료년월'].min()} ~ {by_month['진료년월'].max()})")
    L.append(f"- 환산계수(추정): 금액 {scale_amt:g}, 수량 {scale_qty:g}")
    L.append("")
    L.append("## 1. 전체 합산 비교 (2015~2016)\n")
    L.append("| 항목 | meft(환산) | sick(API) | 커버리지 | 미귀속률 |")
    L.append("|---|---:|---:|---:|---:|")
    L.append(f"| 금액 | {fmt_num(base_amt_sum)} | {fmt_num(api_amt_sum)} | {fmt_pct(cov_all_amt)} | "
             f"{fmt_pct(1-cov_all_amt) if pd.notna(cov_all_amt) else '-'} |")
    L.append(f"| 수량 | {fmt_num(base_qty_sum)} | {fmt_num(api_qty_sum)} | {fmt_pct(cov_all_qty)} | "
             f"{fmt_pct(1-cov_all_qty) if pd.notna(cov_all_qty) else '-'} |")
    L.append("")
    L.append("## 2. 분류코드별 합산 커버리지 (상위 %d)\n" % top)
    L.append("| 분류 | meft금액 | sick금액 | 금액커버리지 | 미귀속률 | 보유비중 | meft월 | sick월 |")
    L.append("|---|---:|---:|---:|---:|---:|---:|---:|")
    for _, r in by_code.head(top).iterrows():
        L.append(f"| {r['code']} | {fmt_num(r['base_amt_won'])} | {fmt_num(r['api_amt'])} "
                 f"| {fmt_pct(r['cov_amt'])} | {fmt_pct(r['unattributed_amt'])} "
                 f"| {fmt_pct(r['share_of_base'])} | {int(r['months_base'])} | {int(r['months_api'])} |")
    L.append("")
    cov = by_code["cov_amt"].dropna()
    bins = [-np.inf, 0.5, 0.9, 0.98, 1.0001, np.inf]
    labels = ["<50%", "50~90%", "90~98%", "98~100%", ">100%"]
    cnt = pd.cut(cov, bins=bins, labels=labels, right=False).value_counts()
    L.append("### 커버리지 분포\n")
    L.append("| 구간 | 분류 수 |")
    L.append("|---|---:|")
    for lab in labels:
        L.append(f"| {lab} | {int(cnt.get(lab, 0))} |")
    L.append("")
    L.append("### 미커버 금액 기여 상위 15개\n")
    L.append("| 분류 | 미귀속률 | 보유비중 | 미커버×보유 | sick금액 |")
    L.append("|---|---:|---:|---:|---:|")
    for _, r in by_code.sort_values("uncovered_share_of_base", ascending=False).head(15).iterrows():
        L.append(f"| {r['code']} | {fmt_pct(r['unattributed_amt'])} | {fmt_pct(r['share_of_base'])} "
                 f"| {r['uncovered_share_of_base']:.3%} | {fmt_num(r['api_amt'])} |")
    L.append("")
    L.append("## 3. 월별 커버리지\n")
    L.append("| 진료년월 | meft금액 | sick금액 | 금액커버리지 | 수량커버리지 |")
    L.append("|---|---:|---:|---:|---:|")
    for _, r in by_month.iterrows():
        L.append(f"| {r['진료년월']} | {fmt_num(r['base_amt_won'])} | {fmt_num(r['api_amt'])} "
                 f"| {fmt_pct(r['cov_amt'])} | {fmt_pct(r['cov_qty'])} |")
    L.append("")

    (out / "audit_report.md").write_text("\n".join(L), encoding="utf-8")

    print("\n".join(L))
    print(f"\n[저장] {out}/audit_report.md")
    print(f"[저장] {out}/audit_by_class.csv")
    print(f"[저장] {out}/audit_by_month.csv")

    return by_code, by_month


# ---------------- 실행 ----------------
by_code, by_month = run(
    api_dir=".",                 # src 자체 → ./sick/data_*.pq
    csv="meft_2014-2025.csv",
    out="audit_",            # 출력 폴더도 1516으로
)
by_code.head(20)

[탐색] pq 100개:
    sick/data_201501.pq
    sick/data_201502.pq
    sick/data_201503.pq
    sick/data_201504.pq
    sick/data_201505.pq
    sick/data_201506.pq
    sick/data_201507.pq
    sick/data_201508.pq
    sick/data_201509.pq
    sick/data_201510.pq
    sick/data_201511.pq
    sick/data_201512.pq
    sick/data_201601.pq
    sick/data_201602.pq
    sick/data_201603.pq
    sick/data_201604.pq
    sick/data_201605.pq
    sick/data_201606.pq
    sick/data_201607.pq
    sick/data_201608.pq
    sick/data_201609.pq
    sick/data_201610.pq
    sick/data_201611.pq
    sick/data_201612.pq
    sick/data_201909.pq
    sick/data_201910.pq
    sick/data_201911.pq
    sick/data_201912.pq
    sick/data_202001.pq
    sick/data_202002.pq
    sick/data_202003.pq
    sick/data_202004.pq
    sick/data_202005.pq
    sick/data_202006.pq
    sick/data_202007.pq
    sick/data_202008.pq
    sick/data_202009.pq
    sick/data_202010.pq
    sick/data_202011.pq
    sick/data_202012.pq
    sick/data_202101.pq
  

,code,base_amt_won,api_amt,base_qty_std,api_qty,months_base,months_api,api_rows,cov_amt,cov_qty,unattributed_amt,share_of_base,uncovered_share_of_base
28,218,2.030593e+13,1.955585e+13,2.886271e+10,2.772929e+10,96,96,233751.0,0.963061,0.960731,0.036939,0.102513,0.003787
92,421,1.888827e+13,1.750344e+13,1.201939e+09,1.135587e+09,96,96,126944.0,0.926683,0.944796,0.073317,0.095356,0.006991
24,214,1.542201e+13,1.493681e+13,2.842004e+10,2.745511e+10,96,96,242249.0,0.968538,0.966048,0.031462,0.077857,0.002450
37,232,1.086041e+13,1.049990e+13,3.914197e+10,3.789918e+10,96,96,306657.0,0.966805,0.968249,0.033195,0.054828,0.001820
89,396,9.976227e+12,9.600698e+12,2.398750e+10,2.307420e+10,96,96,218983.0,0.962358,0.961926,0.037642,0.050364,0.001896
8,119,9.632017e+12,9.267777e+12,1.616742e+10,1.550032e+10,96,96,232874.0,0.962185,0.958738,0.037815,0.048626,0.001839
104,618,7.594268e+12,7.213569e+12,1.710623e+10,1.674094e+10,96,96,279365.0,0.949870,0.978646,0.050130,0.038339,0.001922
4,114,7.193664e+12,7.000006e+12,4.235746e+10,4.127586e+10,96,96,303203.0,0.973079,0.974465,0.026921,0.036317,0.000978
109,629,6.815371e+12,6.454120e+12,3.164077e+09,3.075628e+09,96,96,230682.0,0.946995,0.972046,0.053005,0.034407,0.001824
29,219,6.255586e+12,5.982575e+12,1.988371e+10,1.906692e+10,96,96,252653.0,0.956357,0.958921,0.043643,0.031581,0.001378
